# Nifty50 SMA Crossover Strategy

Simple long-only SMA crossover backtest on the processed Nifty50 minute data.

The strategy goes long when the fast SMA is above the slow SMA. Positions are shifted by one candle so the backtest trades only after the crossover is known.

In [3]:
from pathlib import Path
import os

os.environ.setdefault("PANDAS_USE_NUMEXPR", "0")
os.environ.setdefault("PANDAS_USE_BOTTLENECK", "0")

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
pd.options.display.float_format = "{:,.4f}".format

## Load Processed Nifty50 Data

In [4]:
processed_dir = PROJECT_ROOT / "data" / "processed"

csv_candidates = sorted(processed_dir.glob("nifty50_with_future_volume_*.csv"))
if not csv_candidates:
    csv_candidates = sorted(processed_dir.glob("nifty50_*.csv"))

if not csv_candidates:
    raise FileNotFoundError(f"No processed Nifty50 CSV files found in {processed_dir}")

CSV_PATH = csv_candidates[-1]

df = pd.read_csv(CSV_PATH, parse_dates=["timestamp"])
df = df.sort_values("timestamp").drop_duplicates("timestamp").set_index("timestamp")

required_columns = {"open", "high", "low", "close"}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"Missing required column(s): {sorted(missing_columns)}")

price_columns = ["open", "high", "low", "close"]
df[price_columns] = df[price_columns].apply(pd.to_numeric, errors="coerce")
if "volume" in df.columns:
    df["volume"] = pd.to_numeric(df["volume"], errors="coerce").fillna(0)

df = df.dropna(subset=price_columns)

print(f"Loaded {CSV_PATH.relative_to(PROJECT_ROOT)}")
print(f"{len(df):,} rows from {df.index.min()} to {df.index.max()}")
display(df.head())

Loaded data/processed/nifty50_with_future_volume_2026-01-01_2026-04-30.csv
29,625 rows from 2026-01-01 09:15:00+05:30 to 2026-04-29 15:29:00+05:30


,open,high,low,close,nifty_volume,nifty_open_interest,future_volume,future_open_interest,volume
timestamp,,,,,,,,,
2026-01-01 09:15:00+05:30,"26,173.3000","26,195.3500","26,163.1000","26,183.5000",0,0,61815,14046305,61815
2026-01-01 09:16:00+05:30,"26,181.3500","26,195.1500","26,165.5000","26,194.3000",0,0,34710,14046305,34710
2026-01-01 09:17:00+05:30,"26,193.3500","26,193.3500","26,182.1000","26,189.5500",0,0,24115,14046305,24115
2026-01-01 09:18:00+05:30,"26,190.3500","26,190.3500","26,178.8000","26,179.5000",0,0,23075,14061710,23075
2026-01-01 09:19:00+05:30,"26,178.8500","26,178.8500","26,165.4500","26,167.8000",0,0,27950,14061710,27950


## Strategy Settings

In [5]:
FAST_SMA = 20
SLOW_SMA = 50
INITIAL_CAPITAL = 100_000
COST_PER_TRADE_PCT = 0.0002
MINUTES_PER_YEAR = 252 * 375

if FAST_SMA >= SLOW_SMA:
    raise ValueError("FAST_SMA should be lower than SLOW_SMA for this crossover setup.")

## Build Signals And Backtest

In [6]:
strategy = df.copy()
strategy["fast_sma"] = strategy["close"].rolling(FAST_SMA, min_periods=FAST_SMA).mean()
strategy["slow_sma"] = strategy["close"].rolling(SLOW_SMA, min_periods=SLOW_SMA).mean()

strategy["raw_signal"] = (strategy["fast_sma"] > strategy["slow_sma"]).astype(int)
strategy.loc[strategy["slow_sma"].isna(), "raw_signal"] = 0

strategy["position"] = strategy["raw_signal"].shift(1).fillna(0).astype(int)
strategy["market_return"] = strategy["close"].pct_change().fillna(0)
strategy["turnover"] = strategy["position"].diff().abs().fillna(strategy["position"].abs())
strategy["strategy_return"] = (
    strategy["position"] * strategy["market_return"]
    - strategy["turnover"] * COST_PER_TRADE_PCT
)

strategy["equity"] = INITIAL_CAPITAL * (1 + strategy["strategy_return"]).cumprod()
strategy["buy_hold_equity"] = INITIAL_CAPITAL * (1 + strategy["market_return"]).cumprod()
strategy["drawdown"] = strategy["equity"] / strategy["equity"].cummax() - 1

position_change = strategy["position"].diff().fillna(strategy["position"])
strategy["entry"] = position_change.eq(1)
strategy["exit"] = position_change.eq(-1)

display(strategy.tail())

,open,high,low,close,nifty_volume,nifty_open_interest,future_volume,future_open_interest,volume,fast_sma,...,raw_signal,position,market_return,turnover,strategy_return,equity,buy_hold_equity,drawdown,entry,exit
timestamp,,,,,,,,,,,,,,,,,,,,,
2026-04-29 15:25:00+05:30,"24,157.6500","24,165.5000","24,155.5000","24,157.2000",0,0,33410,14643915,33410,"24,171.5425",...,0,0,-0.0001,0.0000,-0.0000,"87,974.4643","92,261.1568",-0.1218,False,False
2026-04-29 15:26:00+05:30,"24,155.1500","24,155.1500","24,142.9500","24,147.4500",0,0,36530,14643915,36530,"24,169.1800",...,0,0,-0.0004,0.0000,-0.0000,"87,974.4643","92,223.9196",-0.1218,False,False
2026-04-29 15:27:00+05:30,"24,149.2000","24,157.9000","24,148.9500","24,155.9500",0,0,23985,14643915,23985,"24,167.2625",...,0,0,0.0004,0.0000,0.0000,"87,974.4643","92,256.3828",-0.1218,False,False
2026-04-29 15:28:00+05:30,"24,156.4500","24,161.9000","24,151.6000","24,161.1500",0,0,30745,14643915,30745,"24,165.3825",...,0,0,0.0002,0.0000,0.0000,"87,974.4643","92,276.2427",-0.1218,False,False
2026-04-29 15:29:00+05:30,"24,159.7000","24,169.4500","24,157.4000","24,163.6000",0,0,45825,14631305,45825,"24,163.9400",...,0,0,0.0001,0.0000,0.0000,"87,974.4643","92,285.5997",-0.1218,False,False


## Performance Summary

In [7]:
trades = []
entry_time = None
entry_price = None

for ts, row in strategy.loc[strategy["entry"] | strategy["exit"]].iterrows():
    if row["entry"]:
        entry_time = ts
        entry_price = row["close"]
    elif row["exit"] and entry_time is not None:
        exit_time = ts
        exit_price = row["close"]
        gross_return = exit_price / entry_price - 1
        trades.append(
            {
                "entry_time": entry_time,
                "exit_time": exit_time,
                "entry_price": entry_price,
                "exit_price": exit_price,
                "gross_return": gross_return,
                "net_return": gross_return - (2 * COST_PER_TRADE_PCT),
                "duration": exit_time - entry_time,
                "status": "closed",
            }
        )
        entry_time = None
        entry_price = None

if entry_time is not None:
    exit_time = strategy.index[-1]
    exit_price = strategy["close"].iloc[-1]
    gross_return = exit_price / entry_price - 1
    trades.append(
        {
            "entry_time": entry_time,
            "exit_time": exit_time,
            "entry_price": entry_price,
            "exit_price": exit_price,
            "gross_return": gross_return,
            "net_return": gross_return - COST_PER_TRADE_PCT,
            "duration": exit_time - entry_time,
            "status": "open",
        }
    )

trade_log = pd.DataFrame(trades)

years = len(strategy) / MINUTES_PER_YEAR
total_return = strategy["equity"].iloc[-1] / INITIAL_CAPITAL - 1
buy_hold_return = strategy["buy_hold_equity"].iloc[-1] / INITIAL_CAPITAL - 1
annual_return = (1 + total_return) ** (1 / years) - 1 if years > 0 else np.nan
annual_volatility = strategy["strategy_return"].std(ddof=0) * np.sqrt(MINUTES_PER_YEAR)
sharpe = (
    strategy["strategy_return"].mean() / strategy["strategy_return"].std(ddof=0) * np.sqrt(MINUTES_PER_YEAR)
    if strategy["strategy_return"].std(ddof=0) > 0
    else np.nan
)
max_drawdown = strategy["drawdown"].min()
win_rate = (trade_log["net_return"] > 0).mean() if not trade_log.empty else np.nan
avg_trade = trade_log["net_return"].mean() if not trade_log.empty else np.nan

def pct(value):
    return "n/a" if pd.isna(value) else f"{value:.2%}"

def num(value):
    return "n/a" if pd.isna(value) else f"{value:,.2f}"

summary = pd.DataFrame(
    [
        ("Fast SMA", FAST_SMA),
        ("Slow SMA", SLOW_SMA),
        ("Total return", pct(total_return)),
        ("Buy and hold return", pct(buy_hold_return)),
        ("Annualized return", pct(annual_return)),
        ("Annualized volatility", pct(annual_volatility)),
        ("Sharpe ratio", num(sharpe)),
        ("Max drawdown", pct(max_drawdown)),
        ("Trades", len(trade_log)),
        ("Win rate", pct(win_rate)),
        ("Average trade", pct(avg_trade)),
    ],
    columns=["Metric", "Value"],
)

display(summary)
display(trade_log.tail(10))

,Metric,Value
0,Fast SMA,20
1,Slow SMA,50
2,Total return,-12.03%
3,Buy and hold return,-7.71%
4,Annualized return,-33.55%
5,Annualized volatility,13.53%
6,Sharpe ratio,-2.95
7,Max drawdown,-12.85%
8,Trades,337
9,Win rate,25.52%


,entry_time,exit_time,entry_price,exit_price,gross_return,net_return,duration,status
327,2026-04-27 12:08:00+05:30,2026-04-27 14:09:00+05:30,"24,038.6500","24,109.8000",0.0030,0.0026,0 days 02:01:00,closed
328,2026-04-27 14:11:00+05:30,2026-04-27 14:26:00+05:30,"24,116.4000","24,106.8500",-0.0004,-0.0008,0 days 00:15:00,closed
329,2026-04-27 14:57:00+05:30,2026-04-27 15:16:00+05:30,"24,123.3500","24,091.8500",-0.0013,-0.0017,0 days 00:19:00,closed
330,2026-04-28 09:32:00+05:30,2026-04-28 10:44:00+05:30,"24,104.6000","24,134.8500",0.0013,0.0009,0 days 01:12:00,closed
331,2026-04-28 11:03:00+05:30,2026-04-28 11:18:00+05:30,"24,160.8000","24,130.0500",-0.0013,-0.0017,0 days 00:15:00,closed
332,2026-04-28 13:00:00+05:30,2026-04-28 13:36:00+05:30,"24,003.8500","23,976.4500",-0.0011,-0.0015,0 days 00:36:00,closed
333,2026-04-28 14:36:00+05:30,2026-04-28 15:11:00+05:30,"24,020.9000","23,972.6000",-0.0020,-0.0024,0 days 00:35:00,closed
334,2026-04-29 09:21:00+05:30,2026-04-29 11:33:00+05:30,"24,099.5000","24,278.1500",0.0074,0.0070,0 days 02:12:00,closed
335,2026-04-29 12:12:00+05:30,2026-04-29 13:15:00+05:30,"24,302.1000","24,318.5000",0.0007,0.0003,0 days 01:03:00,closed
336,2026-04-29 14:51:00+05:30,2026-04-29 15:05:00+05:30,"24,202.0500","24,191.5000",-0.0004,-0.0008,0 days 00:14:00,closed


## Price Chart With SMA Signals

In [8]:
PLOT_BARS = 2_000
plot_df = strategy.tail(PLOT_BARS).copy()
entry_points = plot_df.loc[plot_df["entry"]]
exit_points = plot_df.loc[plot_df["exit"]]

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    row_heights=[0.78, 0.22],
    vertical_spacing=0.05,
    subplot_titles=(f"Close Price With {FAST_SMA}/{SLOW_SMA} SMA Signals", "Volume"),
)

fig.add_trace(
    go.Scatter(x=plot_df.index, y=plot_df["close"], name="Close", line=dict(color="#1f77b4", width=1.4)),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(x=plot_df.index, y=plot_df["fast_sma"], name=f"SMA {FAST_SMA}", line=dict(color="#2ca02c", width=1.1)),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(x=plot_df.index, y=plot_df["slow_sma"], name=f"SMA {SLOW_SMA}", line=dict(color="#ff7f0e", width=1.1)),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=entry_points.index,
        y=entry_points["close"],
        mode="markers",
        name="Entry",
        marker=dict(symbol="triangle-up", color="#00a676", size=10, line=dict(width=1, color="white")),
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=exit_points.index,
        y=exit_points["close"],
        mode="markers",
        name="Exit",
        marker=dict(symbol="triangle-down", color="#d62728", size=10, line=dict(width=1, color="white")),
    ),
    row=1,
    col=1,
)

if "volume" in plot_df.columns:
    fig.add_trace(
        go.Bar(x=plot_df.index, y=plot_df["volume"], name="Volume", marker_color="rgba(90, 90, 90, 0.35)"),
        row=2,
        col=1,
    )
else:
    fig.add_trace(
        go.Scatter(x=plot_df.index, y=plot_df["position"], name="Position", line_shape="hv"),
        row=2,
        col=1,
    )

fig.update_layout(
    height=760,
    hovermode="x unified",
    template="plotly_white",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)
fig.update_xaxes(rangeslider_visible=False)
fig.update_yaxes(title_text="Price", row=1, col=1)
fig.update_yaxes(title_text="Volume", row=2, col=1)
fig.show()

## One-Day Candlestick SMA View

In [12]:
TARGET_DATE = "2026-03-16"

day_df = strategy.loc[strategy.index.strftime("%Y-%m-%d") == TARGET_DATE].copy()
if day_df.empty:
    raise ValueError(f"No candles found for {TARGET_DATE}")

day_entries = day_df.loc[day_df["entry"]]
day_exits = day_df.loc[day_df["exit"]]

print(f"{TARGET_DATE}: {len(day_df):,} candles from {day_df.index.min().time()} to {day_df.index.max().time()}")

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    row_heights=[0.78, 0.22],
    vertical_spacing=0.05,
    subplot_titles=(f"Nifty50 Candles With {FAST_SMA}/{SLOW_SMA} SMA - {TARGET_DATE}", "Volume"),
)

fig.add_trace(
    go.Candlestick(
        x=day_df.index,
        open=day_df["open"],
        high=day_df["high"],
        low=day_df["low"],
        close=day_df["close"],
        name="Candles",
        increasing_line_color="#00a676",
        decreasing_line_color="#d62728",
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(x=day_df.index, y=day_df["fast_sma"], name=f"SMA {FAST_SMA}", line=dict(color="#2ca02c", width=1.4)),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(x=day_df.index, y=day_df["slow_sma"], name=f"SMA {SLOW_SMA}", line=dict(color="#ff7f0e", width=1.4)),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=day_entries.index,
        y=day_entries["close"],
        mode="markers",
        name="Entry",
        marker=dict(symbol="triangle-up", color="#007f5f", size=11, line=dict(width=1, color="white")),
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=day_exits.index,
        y=day_exits["close"],
        mode="markers",
        name="Exit",
        marker=dict(symbol="triangle-down", color="#b00020", size=11, line=dict(width=1, color="white")),
    ),
    row=1,
    col=1,
)

if "volume" in day_df.columns:
    fig.add_trace(
        go.Bar(x=day_df.index, y=day_df["volume"], name="Volume", marker_color="rgba(90, 90, 90, 0.35)"),
        row=2,
        col=1,
    )

fig.update_layout(
    height=760,
    hovermode="x unified",
    template="plotly_white",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)
fig.update_xaxes(rangeslider_visible=False)
fig.update_yaxes(title_text="Price", row=1, col=1)
fig.update_yaxes(title_text="Volume", row=2, col=1)
fig.show()

2026-03-16: 375 candles from 09:15:00 to 15:29:00


## Equity Curve And Drawdown

In [10]:
fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    row_heights=[0.68, 0.32],
    vertical_spacing=0.06,
    subplot_titles=("Strategy Equity vs Buy And Hold", "Strategy Drawdown"),
)

fig.add_trace(
    go.Scatter(x=strategy.index, y=strategy["equity"], name="SMA strategy", line=dict(color="#1f77b4", width=1.7)),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(x=strategy.index, y=strategy["buy_hold_equity"], name="Buy and hold", line=dict(color="#6c757d", width=1.4)),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=strategy.index,
        y=strategy["drawdown"] * 100,
        name="Drawdown",
        fill="tozeroy",
        line=dict(color="#d62728", width=1.2),
    ),
    row=2,
    col=1,
)

fig.update_layout(
    height=720,
    hovermode="x unified",
    template="plotly_white",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)
fig.update_yaxes(title_text="Portfolio value", row=1, col=1)
fig.update_yaxes(title_text="Drawdown %", ticksuffix="%", row=2, col=1)
fig.show()